# Experiment 004: 3B base-to-GRPO replication, seeds 1 and 2 (Colab)

S4 stage 1: two base-start GRPO runs under the unchanged frozen values,
identical to the seed-0 cell except the seed and the registered ladder
switch (an OOM fails the arm, it never mutates). Seed 1 completes the
proposal-registered second 3B base-GRPO seed; seed 2 is an exploratory
robustness arm and its config carries that status. Base
evals are reused from the shared 004 directory.

The preflight is legacy-aware: the seed-0 manifest predates some provenance
fields, so it asserts recorded library versions directly, the dataset pins
against the audited revisions, and semantic equality of the seed-0 config
with its version at the recorded execution commit. Comment-only edits are allowed.

~11.3h total on an A100-80GB, resume-safe. If a run OOMs: archive BOTH its directories, results/<run_tag> and
checkpoints/<run_tag>, into failed_attempts per the registered procedure,
so no stale checkpoint can silently resume; no retry at changed settings.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [15]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 448.12    # fill from the resources panel
unit_rate = 6.77     # units per hour
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '004_colab_3b_base_grpo_seeds12', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260803_140948 | cu_before = 448.12
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [6]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime per the 6z rule, the seed-0 cell ran 80GB'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-0dcc069f-cda7-9deb-ffc8-ab7cbae2d18f)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [7]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.77 KiB | 604.00 KiB/s, done.
From https://github.com/oleh-hastov/INM363-IndividualProject-Dissertation
   8353433..87be6d5  main       -> origin/main
Updating 8353433..87be6d5
Fast-forward
 .../notebooks/004_colab_3b_base_grpo_seeds12.ipynb | 91 ++++++----------------
 001_Code/tests/test_7b_pilot.py                    |  1 -
 001_Code/tests/test_s4_base_grpo_replication.py    |  4 +-
 3 files changed, 25 insertions(+), 71 deletions(-)
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [8]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization is used here, so just drop it
!pip uninstall -q -y torchao

# pin the runs to what the seed-0 manifest actually records
import importlib, json, os, subprocess, yaml
ref_path = 'results/004_COLAB_FULL_B3_base_grpo_seed0/manifest.json'
assert os.path.exists(ref_path), 'reference manifest missing, pull the repo with results before training'
ref = json.load(open(ref_path))
for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
    have = importlib.import_module(lib).__version__
    assert have == ref['versions'][lib], f'{lib} {have} differs from the recorded {ref["versions"][lib]}'

# the manifest predates the dataset-revision fields, so the pins are checked
# against the audited revisions instead
from data.mmlu import MMLU_REVISION
from data.gsm8k import GSM8K_REVISION
assert MMLU_REVISION == 'c30699e8356da336a370243923dbaf21066bb9fe'
assert GSM8K_REVISION == '740312add88f781978c0658806c59bc2815b9866'

# the seed-0 config values must still equal the recorded run config
commit = ref['git']['commit']
assert ref['git']['dirty'] is False
old = subprocess.run(['git', 'show', f"{commit}:001_Code/configs/004_colab_full_b3_base_grpo_seed0.yaml"],
                     capture_output=True, text=True)
if old.returncode != 0:
    old = subprocess.run(['git', 'show', f"{commit}:configs/004_colab_full_b3_base_grpo_seed0.yaml"],
                         capture_output=True, text=True)
assert old.returncode == 0, 'could not read the seed-0 config at its run commit'
recorded_cfg = yaml.safe_load(old.stdout)
current_cfg = yaml.safe_load(open('configs/004_colab_full_b3_base_grpo_seed0.yaml'))
assert recorded_cfg == current_cfg, 'seed-0 config values drifted since its recorded run'
print('environment, dataset pins and historical config values verified')

environment, dataset pins and historical config values verified


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [9]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [10]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 20, 'need >=20GB free on drive for the 3b run'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260803_140948.json
drive free: 179 GB


## 6. Build the data splits

In [11]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 20.0MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 2.74MB/s,  226kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 2.74MB/s,  227kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 683kB/s, 41.4kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 682kB/s, 41.4kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 383995.71 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 305364.41 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 86.8MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 149MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 7. Preflight: the shared 004 base evals must exist

In [12]:
import json, os
D = '/content/drive/MyDrive/inm363'
be = os.path.join(D, 'results', 'base_evals', '004_COLAB_FULL_B3')
# the runner skips base evals when these exist, so their completeness is
# checked here rather than discovered after the expensive runs
for f in ('base_eval.json', 'base_logprob.json'):
    fp = os.path.join(be, f)
    assert os.path.exists(fp), f'shared 004 base eval missing: {f}'
    assert json.load(open(fp))['n_items'] == 1884, f'{f}: wrong item count'
for f in ('base_preds.jsonl', 'base_logprob_preds.jsonl'):
    fp = os.path.join(be, f)
    assert os.path.exists(fp), f'shared 004 base eval missing: {f}'
    rows = sum(1 for line in open(fp) if line.strip())
    assert rows == 1884, f'{f}: {rows} rows'
print('shared base evals present and complete, both runs will skip them')

shared base evals present and complete, both runs will skip them


## 8. Wiring check

In [13]:
import subprocess, sys
for s in (1, 2):
    print('wiring: seed', s)
    subprocess.run([sys.executable, 'scripts/run.py', '--config',
                    f'configs/004_colab_full_b3_base_grpo_seed{s}.yaml'], check=True)

wiring: seed 1
wiring: seed 2


## 9. The run

**If colab disconnects:** re-run this cell, finished stages skip.

In [ ]:
# both cost fields must be filled before the long runs start
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
# python loop with ! calls so the logs stream live, exit code checked so a
# failed run stops the session instead of hiding behind the next one
for cfg in ('004_colab_full_b3_base_grpo_seed1', '004_colab_full_b3_base_grpo_seed2'):
    print(f'===== {cfg} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{cfg}.yaml --stage all
    if _exit_code != 0:
        raise SystemExit(f'{cfg} failed with exit code {_exit_code}; an oom means archive results/<run_tag> AND checkpoints/<run_tag> per the registered procedure, never retry with changed settings')

Streaming output truncated to the last 5000 lines.
│ │               │ change even   │               │              │           │ │
│ │               │ if they think │               │              │           │ │
│ │               │ it does; this │               │              │           │ │
│ │               │ is a          │               │              │           │ │
│ │               │ transcript    │               │              │           │ │
│ │               │ mismatch      │               │              │           │ │
│ │               │ concept. It's │               │              │           │ │
│ │               │ not clear     │               │              │           │ │
│ │               │ that Kate is  │               │              │           │ │
│ │               │ demonstrating │               │              │           │ │
│ │               │ it. The terms │               │              │           │ │
│ │               │ animism and   │               │       

## 10. Peek at the results

In [ ]:
import json, os
D = '/content/drive/MyDrive/inm363'
for tag in ('004_COLAB_FULL_B3_base_grpo_seed0', '004_COLAB_FULL_B3_base_grpo_seed1',
            '004_COLAB_FULL_B3_base_grpo_seed2'):
    sm = os.path.join(D, 'results', tag, 'summary.json')
    if os.path.exists(sm):
        j = json.load(open(sm))
        print(f"{tag}: grpo {j['grpo_acc']}")
    else:
        print(f'{tag}: no summary yet')

004_COLAB_FULL_B3_base_grpo_seed0: grpo {'easy': 0.7781569965870307, 'medium': 0.6174055829228243, 'hard': 0.5429292929292929}
004_COLAB_FULL_B3_base_grpo_seed1: grpo {'easy': 0.7747440273037542, 'medium': 0.625615763546798, 'hard': 0.5580808080808081}
004_COLAB_FULL_B3_base_grpo_seed2: grpo {'easy': 0.7667804323094426, 'medium': 0.6288998357963875, 'hard': 0.5378787878787878}


## 11. Close the units record

In [ ]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 354.93     # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
